# FlashAttention과 다른 메모리 절감 방법 비교

이 노트북은 학습 모델 없이 **Attention score 메모리**만 비교합니다.

비교 대상은 다음과 같습니다.

1. Larger patch
2. Window attention
3. Shifted window
4. Patch merging
5. FlashAttention
6. Temporal chunking
7. FP16/BF16

> Activation checkpointing은 학습 모델의 중간 activation이 있어야 의미가 있으므로 이번 비교에서 제외했습니다.

# 챕터 1. 비교 기준

localizer_first_level 데이터에서 확인한 형상을 사용합니다.

| 항목 | 값 |
|---|---:|
| 공간 크기 | 53 × 63 × 46 |
| 전체 시간 길이 | 128 frames |
| 기준 시간 길이 | 20 frames |
| 기준 spatial patch | 6 × 6 × 6 |
| 기준 dtype | FP16, 2 bytes |
| batch / head / layer | 각각 1 |

기준 token grid는 (9, 11, 8, 20), token 수 N은 15,840입니다.

Dense attention의 score 메모리는 다음과 같습니다.

$$\text{Score memory}=N^2\times\text{bytes per value}$$

이 비교는 **N² score tensor 하나의 저장량**을 기준으로 합니다. 모델 전체 VRAM은 포함하지 않습니다.

In [1]:
import math
import pandas as pd

spatial_shape = (53, 63, 46)
base_frames = 20
base_patch = 6
fp16_bytes = 2

def dense_score(patch, frames, value_bytes=2):
    grid = tuple(
        math.ceil(size / patch) for size in spatial_shape
    ) + (frames,)
    tokens = math.prod(grid)
    memory_mb = tokens**2 * value_bytes / 1e6
    return grid, tokens, memory_mb

def window_score(grid, window, value_bytes=2):
    window_count = math.prod(
        math.ceil(g / w) for g, w in zip(grid, window)
    )
    tokens_per_window = math.prod(window)
    memory_mb = (
        window_count * tokens_per_window**2
        * value_bytes / 1e6
    )
    return memory_mb

base_grid, base_tokens, baseline_mb = dense_score(
    base_patch, base_frames, fp16_bytes
)

baseline_mb

501.8112

# 챕터 2. 방법별 계산 기준

## Token 수 또는 attention 범위를 줄이는 방법

- **Larger patch:** patch를 6³에서 12³으로 키웁니다.
- **Window attention:** 전체 N² 대신 각 window 안의 score만 저장합니다.
- **Shifted window:** score 크기는 같은 크기의 일반 window와 동일합니다.
- **Patch merging:** spatial token grid를 단계마다 절반으로 줄입니다.
- **Temporal chunking:** 128 frames를 한 번에 처리하지 않고 20 frames씩 처리합니다.

## 원소 크기를 줄이는 방법

- **FP16/BF16:** FP32의 4 bytes를 2 bytes로 줄입니다.

## N² score를 저장하지 않는 방법

- **FlashAttention:** token 수 N과 attention 범위는 유지하지만 N² score 전체를 메모리에 만들지 않습니다.
- 따라서 score tensor 저장량 관점에서는 기준 501.811 MB의 할당을 피합니다.
- 단, Q/K/V, 출력, softmax 통계와 kernel workspace는 남으므로 **전체 attention 메모리가 0 MB라는 뜻은 아닙니다.**

In [2]:
# Larger patch
_, _, patch12_mb = dense_score(12, 20, fp16_bytes)

# Window / shifted window
window4_mb = window_score(base_grid, (4, 4, 4, 4))
window2_mb = window_score(base_grid, (2, 2, 2, 2))

# Patch merging
merge_grid = base_grid
merge_memory = []
for stage in range(1, 4):
    merge_grid = tuple(
        math.ceil(g / 2) for g in merge_grid[:3]
    ) + (merge_grid[3],)
    merge_tokens = math.prod(merge_grid)
    merge_mb = merge_tokens**2 * fp16_bytes / 1e6
    merge_memory.append(merge_mb)

# Temporal chunking: 128 frames 전체와 20 frames peak 비교
_, _, full128_mb = dense_score(6, 128, fp16_bytes)

# FP32와 FP16 비교
_, _, baseline_fp32_mb = dense_score(6, 20, 4)

rows = [
    ['Dense FP16 기준', f'{baseline_mb:,.3f}', '1배', '비교 기준'],
    ['Larger patch 12³', f'{patch12_mb:,.3f}', f'{baseline_mb / patch12_mb:,.2f}배', '세부 공간 정보 감소'],
    ['Window 4⁴', f'{window4_mb:,.3f}', f'{baseline_mb / window4_mb:,.2f}배', '장거리 직접 연결 제한'],
    ['Window 2⁴', f'{window2_mb:,.3f}', f'{baseline_mb / window2_mb:,.2f}배', '장거리 직접 연결이 더 제한됨'],
    ['Shifted window 4⁴', f'{window4_mb:,.3f}', f'{baseline_mb / window4_mb:,.2f}배', 'shift mask 필요'],
    ['Patch merging 1회', f'{merge_memory[0]:,.3f}', f'{baseline_mb / merge_memory[0]:,.2f}배', '공간 정보 압축'],
    ['Patch merging 2회', f'{merge_memory[1]:,.3f}', f'{baseline_mb / merge_memory[1]:,.2f}배', '공간 정보 추가 압축'],
    ['Patch merging 3회', f'{merge_memory[2]:,.4f}', f'{baseline_mb / merge_memory[2]:,.0f}배', '매우 강한 공간 압축'],
    ['FlashAttention', 'N² score 미저장', f'{baseline_mb:,.3f} MB 할당 회피', '지원 GPU와 kernel 필요'],
    ['128 → 20-frame chunk', f'{baseline_mb:,.3f} peak', f'{full128_mb / baseline_mb:,.2f}배', 'chunk 사이 관계 단절'],
    ['FP32 → FP16/BF16', f'{baseline_fp32_mb:,.3f} → {baseline_mb:,.3f}', '2.00배', '정밀도 저하 가능'],
]

comparison_df = pd.DataFrame(
    rows,
    columns=['방법', '물질화되는 score memory (MB)', '기준 대비 절감', '비용 또는 단점']
)
comparison_df.style.set_properties(
    **{'text-align': 'left'}
).hide(axis='index')

방법,물질화되는 score memory (MB),기준 대비 절감,비용 또는 단점
Dense FP16 기준,501.811,1배,비교 기준
Larger patch 12³,11.520,43.56배,세부 공간 정보 감소
Window 4⁴,11.796,42.54배,장거리 직접 연결 제한
Window 2⁴,0.614,816.75배,장거리 직접 연결이 더 제한됨
Shifted window 4⁴,11.796,42.54배,shift mask 필요
Patch merging 1회,11.520,43.56배,공간 정보 압축
Patch merging 2회,0.259,"1,936.00배",공간 정보 추가 압축
Patch merging 3회,0.0128,"39,204배",매우 강한 공간 압축
FlashAttention,N² score 미저장,501.811 MB 할당 회피,지원 GPU와 kernel 필요
128 → 20-frame chunk,501.811 peak,40.96배,chunk 사이 관계 단절


# 챕터 3. FlashAttention을 같은 표에서 해석하는 방법

FlashAttention은 다른 방법과 메모리를 줄이는 방식이 다릅니다.

| 구분 | Larger patch / window / merging / chunk | FlashAttention |
|---|---|---|
| Token 수 N | 감소하거나 지역 범위로 제한 | 유지 |
| Attention 범위 | 줄어들 수 있음 | 유지 |
| N² score 전체 저장 | 더 작은 크기로 저장 | 저장하지 않음 |
| 정보 손실 가능성 | 구조에 따라 있음 | attention 범위 자체는 유지 |
| 정확한 peak 계산 | 식으로 score 크기 계산 가능 | GPU kernel 실측 필요 |
| 주요 비용 | 해상도 또는 연결 범위 감소 | kernel 지원 필요, N² 연산량은 남음 |

따라서 score 저장량만 보면 FlashAttention이 501.811 MB의 N² tensor 할당을 피합니다. 그러나 다른 방법은 N 자체 또는 attention 범위를 줄이므로 **연산량도 함께 줄일 수 있다는 장점**이 있습니다.

# 챕터 4. 현재 CUDA 상태

아래 셀은 현재 PyTorch가 CUDA를 사용할 수 있는지 확인합니다.

CUDA 지원 PyTorch가 설치된 환경에서는 학습 모델 없이 임의의 Q/K/V tensor만으로도 Manual attention과 FlashAttention의 실제 peak VRAM을 비교할 수 있습니다.

In [3]:
import torch
import torch.nn.functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel

cuda_ready = torch.cuda.is_available()
flash_ready = (
    cuda_ready
    and torch.backends.cuda.is_flash_attention_available()
)

pd.DataFrame({
    '항목': ['PyTorch 버전', 'PyTorch build', 'CUDA 사용 가능', 'FlashAttention kernel'],
    '결과': [
        torch.__version__,
        'CUDA build' if torch.version.cuda else 'CPU-only build',
        str(cuda_ready),
        '사용 가능' if flash_ready else '사용 불가'
    ]
})

,항목,결과
0,PyTorch 버전,2.14.1+cu130
1,PyTorch build,CUDA build
2,CUDA 사용 가능,True
3,FlashAttention kernel,사용 가능


# 챕터 5. 실제 GPU peak 측정

이 셀은 CUDA와 FlashAttention kernel이 모두 있을 때만 실행됩니다.

- 실제 fMRI 계산 기준과 같은 token 수 N=15,840, head 1, head dimension 64를 사용합니다.
- Manual attention은 N² score와 softmax 결과를 직접 만듭니다.
- FlashAttention은 PyTorch SDPA에서 Flash backend를 강제로 사용합니다.
- 학습된 모델은 필요하지 않으며 같은 크기의 임의 Q/K/V tensor로 비교합니다.

In [4]:
def measure_gpu_peak(mode, tokens=15840, heads=1, head_dim=64):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    shape = (1, heads, tokens, head_dim)
    q = torch.randn(shape, device='cuda', dtype=torch.float16, requires_grad=True)
    k = torch.randn(shape, device='cuda', dtype=torch.float16, requires_grad=True)
    v = torch.randn(shape, device='cuda', dtype=torch.float16, requires_grad=True)

    if mode == 'manual':
        score = q @ k.transpose(-2, -1)
        score = score / math.sqrt(head_dim)
        output = score.softmax(dim=-1) @ v
    else:
        with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
            output = F.scaled_dot_product_attention(q, k, v)

    output.float().square().mean().backward()
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / 1e6


if flash_ready:
    manual_peak = measure_gpu_peak('manual')
    flash_peak = measure_gpu_peak('flash')

    display(pd.DataFrame([
        ['Manual attention', manual_peak, 1.0],
        ['FlashAttention', flash_peak, manual_peak / flash_peak],
    ], columns=['방법', 'CUDA peak allocation (MB)', '절감 배수']).style.format({
        'CUDA peak allocation (MB)': '{:,.3f}',
        '절감 배수': '{:,.2f}배'
    }).hide(axis='index'))
else:
    print('현재 PyTorch가 CPU-only build라 GPU 실측을 건너뜁니다.')

방법,CUDA peak allocation (MB),절감 배수
Manual attention,"2,536.521",1.00배
FlashAttention,46.064,55.07배


# 챕터 6. 결론

- **가장 직접적인 score 저장 제거:** FlashAttention
- **연산량까지 크게 줄이기 쉬운 방법:** Window attention, larger patch, patch merging
- **구현이 가장 단순한 고정 비율 절감:** FP16/BF16
- **긴 fMRI의 peak를 조절하기 쉬운 방법:** Temporal chunking

FlashAttention은 attention 범위를 유지하면서 N² score 저장을 피한다는 장점이 있습니다. 반면 N² 연산 자체는 남으므로, 매우 긴 fMRI에서는 window attention이나 token 감소 방법과 함께 검토하는 것이 합리적입니다.

## RTX 4060 실측 결과

실제 fMRI 기준인 N=15,840, head 1, head dimension 64, FP16에서 forward와 backward의 PyTorch peak allocation을 측정했습니다.

- Manual attention: **2,536.521 MB**
- FlashAttention: **46.064 MB**
- 실측 peak allocation 감소: **약 55.07배**

따라서 FlashAttention은 이 조건에서 501.811 MB의 score tensor 자체를 물질화하지 않을 뿐 아니라, softmax와 역전파에 필요한 대형 중간 tensor까지 피하여 전체 peak allocation을 크게 줄였습니다. 이 결과는 RTX 4060과 현재 PyTorch/CUDA 환경에서 측정한 값이므로 다른 GPU나 shape에서는 달라질 수 있습니다.